# Binding energy between CO₂ and Propionitrile molecule

In [ ]:
!service mongodb start

In [ ]:
%env FW_CONFIG_FILE=/root/capsule/code/config/FW_config.yaml

In [ ]:
import warnings
warnings.filterwarnings('ignore')

This tutorial focuses on the binding energy (BE) workflow in MISPR, which represents the recipe that outlines the steps for how to automate binding energy calculations from start to finish. 

There are many preset workflows in MISPR for calculating common properties. To explore all available workflows, you can do that by checking the MISPR source code linked at: 
1. DFT workflows: https://github.com/molmd/mispr/tree/master/mispr/gaussian/workflows/base
2. MD workflows: https://github.com/molmd/mispr/tree/master/mispr/lammps/workflows

For this tutorial, we will focus on the BE workflow, which is a DFT-based calculation. Binding energy is the amount of energy required to bring two particles or molecules together so that they form a bond. It is a critical property in materials science applications because it allows us to understand the interactions between the species forming our systems, thereby providing a way to tailor the system composition according to the desired degree of interactions. 

BE is calculated in MISPR according to the following equation:

$BE = E_{final} - (E_{mol1} + E_{mol2})$

In the above equation, E is the energy on the final complex as well as the energy on each individual molecules *i*. 

The workflows output the BE in the electron volts (eV) unit, but this can be converted to other units like kcal/mol and kJ/mol. 

# Binding energy between CO₂ and Propionitrile molecules 

# Generating CO₂ structure

We will use the BE workflow in MISPR to calculate the BE between CO₂ and Propionitrile molecules. Let us begin by importing a basic CO₂ structure and taking a look at its structure.

An xyz file is a convenient way to store information about the structure of a molecule. We can examine the contents of an CO₂ file for a CO₂ molecule:

In [ ]:
with open("/root/capsule/data/be/co2.xyz", "r") as f:
    print(f.read())

In [ ]:
with open("/root/capsule/data/be/propionitrile.xyz", "r") as f:
    print(f.read())

We can use the Molecule class from pymatgen to represent the molecule as a python object

In [ ]:
from pymatgen.core.structure import Molecule 

co2_mol = Molecule.from_file("/root/capsule/data/be/co2.xyz")
print(co2_mol)

In [ ]:
from IPython.display import Image
Image("/root/capsule/data/be/Carbon_Dioxide.png", width=600, height=600)

In [ ]:
from pymatgen.core.structure import Molecule 

propionitrile_mol = Molecule.from_file("/root/capsule/data/be/propionitrile.xyz")
print(propionitrile_mol)

In [ ]:
from IPython.display import Image
Image("/root/capsule/data/be/Propionitrile.png", width=900, height=900)

MISPR workflows (with the support of FireWorks library) compiles everything needed to: 

* Take and store workflow inputs from the user 
* Create new calculation directories and write input files 
* Parse output directories and store results in a MongoDB database

# Getting workflows

Now, let's get started by importing the BE workflow and then creating a workflow object from CO₂ and Propionitrile structure.

In [ ]:
import os
from mispr.gaussian.utilities.fw_utilities import run_fake_gaussian
from mispr.gaussian.workflows.base.binding_energy import get_binding_energies

# Define file and working directories
ref_dir = "/root/capsule/data/be"
working_dir = "/root/capsule/results/be"


# Define file paths instead of loading Molecule objects
co2_path = os.path.join(ref_dir, "co2.xyz")
propionitrile_path = os.path.join(ref_dir, "propionitrile.xyz")

# Create the binding energy workflow
wf = get_binding_energies(
    mol_operation_type=["get_from_mol", "get_from_mol"],
    #mol_operation_type=["get_from_file", "get_from_file"],
    mol=[propionitrile_mol, co2_mol],
    #mol=[co2_path, propionitrile_path],
    index=[3, 2],  
    process_mol_func=False,
    save_to_db=True,
    save_to_file=True,
    mol_name=["propionitrile", "co2"],
    working_dir=working_dir,
    opt_gaussian_inputs={
        "functional": "B3LYP",
        "basis_set": "6-31+G*",
        "route_parameters": {
            "Opt": "(calcfc,tight,cartesian,maxcycles=100)",
            "SCF": "(tight,xqc)",
            "int": "ultrafine",
            "NoSymmetry": None,
            "test": None,
            "EmpiricalDispersion": "GD3",
            "symmetry": "none"
        },
        "link0_parameters": {
            "%chk": "opt.chk",
            "%mem": "45GB",
            "%nprocshared": "28"
        }
    },
    freq_gaussian_inputs={
        "functional": "B3LYP",
        "basis_set": "6-31+G*",
        "route_parameters": {
            "Freq": None,
            "iop(7/33=1)": None,
            "NoSymmetry": None,
            "test": None,
            "EmpiricalDispersion": "GD3",
            
        },
        "link0_parameters": {
            "%chk": "freq.chk",
            "%mem": "45GB",
            "%nprocshared": "28"
        }
    }
)


print(wf)


We can see that this workflow contains seven fireworks, meaning that it is a complex workflow. The exact steps in each firework can vary but for these cases, each firework roughly represents one Gaussian calculation which consists of the following steps:

1. Processing the input molecule 
2. Writing an input file for DFT calculations 
3. Executing commands to run the calculations 
4. Parsing the output files and saving the results in the database 

All these steps are typically done manual and require a lot of effort, but with MISPR, the entire process is simplified to one block of code. 

The workflow will run fireworks for the individual molecules, take the structure of these molecules and create a complex structure, where the molecules bind at a specific site. It will then optimize that structure and calculate its energy. The last firework in this workflow is an analysis step that computes the BE using the equation above. 

Before we move forward with running the BE workflow, we will make one modification called run_fake_gaussian. This will change the fireworks in the workflows so they will not actually run Gaussian (which requires greater computer resources and a Gaussian license), but instead will simulate running Gaussian by simply copying over pre-existing Gaussian output files. However, the overall steps in the workflow remain the same as if you are running the calculations on supercomputing resources. 

In [ ]:
wf = run_fake_gaussian(
    wf,
    ref_dirs=[
        f"{ref_dir}/propionitrile/Optimization",   
        f"{ref_dir}/propionitrile/Frequency",
        f"{ref_dir}/co2/Optimization",             
        f"{ref_dir}/co2/Frequency",
        f"{ref_dir}/propionitrile_co2/Optimization",
        f"{ref_dir}/propionitrile_co2/Frequency",
    ],
    input_files=[
        "propionitrile_opt.com",
        "propionitrile_freq.com",
        "co2_opt.com",
        "co2_freq.com",
        "propionitrile_co2_opt.com",
        "propionitrile_co2_freq.com",
    ],
    tolerance=10,
)


### Launching workflows from the database 

Now we will introduce the LaunchPad from FireWorks which is a python object that let's us interact with MongoDB and manage our calculations. The LaunchPad allows you to submit and query workflows from anywhere you have database access. We need to get ourselves a LaunchPad object and set it up so we can submit our workflows.

In [ ]:
from datetime import date
from fireworks import LaunchPad

today = date.today()
d = today.strftime("%Y-%m-%d")

lpad = LaunchPad.auto_load()

lpad.reset(d)

Note that lpad.reset() should only be executed one time when you are first initializing your database. If you reset your LaunchPad at a later time, you will erase the record of past calcuations you have run.

Now we will add our generated workflow object into MongoDB so that it can be accessed later.

In [ ]:
lpad.add_wf(wf)

Now the workflow is added to the database and we can check its status using `.get_wf_summary_dict()`

In [ ]:
lpad.get_wf_summary_dict(fw_id=1)

In [ ]:
from fireworks.core.rocket_launcher import rapidfire

rapidfire(lpad)

# Checking the outputs

Let us check the output file (named binding_energy.json) containing a summary of the calculation results. Typically, users will mostly be interested in this final file instead of the intermediate files generated from running the calculations.

In [ ]:
import json
import numpy as np

with open("/root/capsule/results/be/analysis/binding_energy.json") as json_file:
    data = json.load(json_file)
print(data)

We can also open the file in a text editor and check its entries, but let's find out how much is the binding energy between CO₂ and Propionitrile molecules. 

In [ ]:
be = np.round(data["be_eV"]["value"] * 23, 2)
print(f"Binding energy between CO₂ and Propionitrile molecules is: {be} kcal/mol")

Let us visualize the generated structure. 

In [ ]:
Image("/root/capsule/data/be/co2_propionitrile.png", width=300, height=300, retina=True, unconfined=True)